# HW3: Tiny Video Motion and Consistency Analyzer

Implementations for motion-compensated frame prediction (Section 1), temporal consistency metrics (Section 2), and the hierarchical block matching bonus (Section 4, Option B).

In [1]:
from pathlib import Path
import time

import matplotlib.pyplot as plt
import numpy as np

from hw3_utils import (
    ensure_dir,
    make_blur_clip,
    make_flicker_clip,
    make_jitter_clip,
    make_translation_clip,
    plot_motion_vectors,
    plot_prediction,
    psnr,
    residual_entropy,
    save_gif,
    summarize_prediction,
)

RESULT_DIR = ensure_dir('result')
np.random.seed(0)

## 0. Synthetic clips

We use the four synthetic clips bundled in `hw3_utils.py`: a clean translating square, a flicker variant (alternating frames brighter), a horizontal jitter clip, and a Gaussian-blurred translating square. Each is 12 frames at 128x128 grayscale.

In [2]:
clips = {
    'clean_translation': make_translation_clip(),
    'flicker': make_flicker_clip(),
    'jitter': make_jitter_clip(),
    'blur': make_blur_clip(),
}

for name, clip in clips.items():
    save_gif(clip, RESULT_DIR / f'{name}.gif')

video = clips['clean_translation']
prev_frame = video[0]
cur_frame = video[1]
print(video.shape, video.dtype, video.min(), video.max())

(12, 128, 128) float32 20.0 220.0


## 1. Motion-compensated frame prediction

Section 1.1 implements the previous-frame baseline. Section 1.2 implements exhaustive block matching with SAD or MSE block metrics, returning a `(H/B, W/B, 2)` motion-vector field stored as `(dy, dx)` together with the motion-compensated prediction and residual.

In [3]:
def frame_difference(prev_frame, cur_frame):
    """Previous-frame predictor: pred = prev_frame, residual = cur - prev."""
    prev = np.asarray(prev_frame, dtype=np.float32)
    cur = np.asarray(cur_frame, dtype=np.float32)
    pred = prev.copy()
    residual = cur - pred
    return pred, residual


def block_matching(prev_frame, cur_frame, block_size=16, search_radius=8, metric='sad'):
    """Exhaustive block matching over (u, v) in [-R, R]^2.

    Vectors stored as (dy, dx); the candidate block is sampled from
    prev_frame[y + dy, x + dx]. Blocks whose candidate falls outside the image
    boundary are skipped (those candidate scores are set to +inf).
    """
    prev = np.asarray(prev_frame, dtype=np.float32)
    cur = np.asarray(cur_frame, dtype=np.float32)
    H, W = cur.shape
    B = block_size
    R = search_radius
    Hb, Wb = H // B, W // B

    # Reshape current frame into block grid: (Hb, Wb, B, B)
    cur_blocks = cur[:Hb * B, :Wb * B].reshape(Hb, B, Wb, B).transpose(0, 2, 1, 3)

    best_cost = np.full((Hb, Wb), np.inf, dtype=np.float32)
    best_dy = np.zeros((Hb, Wb), dtype=np.int32)
    best_dx = np.zeros((Hb, Wb), dtype=np.int32)

    # Top-left pixel of each block in the current frame.
    block_y = np.arange(Hb) * B
    block_x = np.arange(Wb) * B

    for dy in range(-R, R + 1):
        for dx in range(-R, R + 1):
            # Per-block validity: candidate top-left in [0, H-B] and [0, W-B].
            ys = block_y + dy
            xs = block_x + dx
            valid_y = (ys >= 0) & (ys + B <= H)
            valid_x = (xs >= 0) & (xs + B <= W)
            if not valid_y.any() or not valid_x.any():
                continue
            valid = np.outer(valid_y, valid_x)

            # Build the candidate-block grid by shifting `prev`.
            # Use clipped indexing for efficiency, then mask invalid entries.
            ys_c = np.clip(ys, 0, H - B)
            xs_c = np.clip(xs, 0, W - B)
            # gather (Hb, Wb, B, B)
            cand = np.empty((Hb, Wb, B, B), dtype=np.float32)
            for ib, y0 in enumerate(ys_c):
                for jb, x0 in enumerate(xs_c):
                    cand[ib, jb] = prev[y0:y0 + B, x0:x0 + B]

            diff = cur_blocks - cand
            if metric == 'sad':
                cost = np.sum(np.abs(diff), axis=(2, 3))
            elif metric == 'mse':
                cost = np.mean(diff * diff, axis=(2, 3))
            else:
                raise ValueError(f'unknown metric {metric!r}')
            cost = np.where(valid, cost, np.inf)

            improved = cost < best_cost
            best_cost = np.where(improved, cost, best_cost)
            best_dy = np.where(improved, dy, best_dy)
            best_dx = np.where(improved, dx, best_dx)

    motion_vectors = np.stack([best_dy, best_dx], axis=-1).astype(np.int32)

    pred = cur.copy()
    for ib in range(Hb):
        for jb in range(Wb):
            dy = int(best_dy[ib, jb])
            dx = int(best_dx[ib, jb])
            y0 = ib * B
            x0 = jb * B
            ys = y0 + dy
            xs = x0 + dx
            if 0 <= ys <= H - B and 0 <= xs <= W - B:
                pred[y0:y0 + B, x0:x0 + B] = prev[ys:ys + B, xs:xs + B]
            else:
                pred[y0:y0 + B, x0:x0 + B] = prev[y0:y0 + B, x0:x0 + B]
    residual = cur - pred
    return motion_vectors, pred, residual

In [4]:
# Section 1.1: previous-frame baseline visualization on the clean translation clip.
pred_base, residual_base = frame_difference(prev_frame, cur_frame)
plot_prediction(prev_frame, cur_frame, pred_base, residual_base, RESULT_DIR / 'baseline_prediction.png')
print('baseline:', summarize_prediction(cur_frame, pred_base, residual_base))

# Sanity check: square moves by (dy=2, dx=4) in the synthetic clip; block-matching
# vectors over the square should be approximately (-2, -4).
mv_check, _, _ = block_matching(prev_frame, cur_frame, block_size=16, search_radius=8, metric='sad')
moved = mv_check.reshape(-1, 2)
moved = moved[(moved != 0).any(axis=1)]
if len(moved):
    print('non-zero MV unique values:', np.unique(moved, axis=0))

baseline: {'mse': 781.25, 'psnr': 19.20290330515779, 'residual_entropy': 0.1583317479797322}
non-zero MV unique values: [[-8 -8]
 [-8  0]
 [-6 -8]
 [-2 -8]
 [-2 -4]
 [ 0 -8]]


In [5]:
# Section 1.2: evaluate (block_size, search_radius) on >= 5 frame pairs from the
# clean translation clip. We also save visualizations for the first pair at every
# (B, R) configuration. Baseline (frame difference) statistics are recorded as a
# reference row per frame pair.
settings = [(8, 4), (16, 4), (16, 8), (32, 8)]
n_pairs = 5
rows = []

clip_name = 'clean_translation'
clip = clips[clip_name]

for t in range(1, 1 + n_pairs):
    prev_t = clip[t - 1]
    cur_t = clip[t]

    # baseline reference (frame difference)
    t0 = time.perf_counter()
    pred_b, res_b = frame_difference(prev_t, cur_t)
    elapsed_b = time.perf_counter() - t0
    stats_b = summarize_prediction(cur_t, pred_b, res_b)
    rows.append({
        'clip': clip_name, 'frame_idx': t, 'method': 'baseline',
        'block_size': None, 'search_radius': None, 'metric': None,
        'runtime_s': elapsed_b, **stats_b,
    })

    for block_size, search_radius in settings:
        for metric in ('sad', 'mse'):
            t0 = time.perf_counter()
            mv, pred, residual = block_matching(prev_t, cur_t, block_size, search_radius, metric=metric)
            elapsed = time.perf_counter() - t0
            stats = summarize_prediction(cur_t, pred, residual)
            rows.append({
                'clip': clip_name, 'frame_idx': t, 'method': 'block',
                'block_size': block_size, 'search_radius': search_radius, 'metric': metric,
                'runtime_s': elapsed, **stats,
            })
            if t == 1 and metric == 'sad':
                tag = f'b{block_size}_r{search_radius}'
                plot_prediction(prev_t, cur_t, pred, residual,
                                RESULT_DIR / f'block_{tag}_pred.png')
                plot_motion_vectors(mv, block_size,
                                    RESULT_DIR / f'block_{tag}_mv.png')

# Print a compact aggregate table per (B, R, metric).
from collections import defaultdict
agg = defaultdict(list)
for r in rows:
    if r['method'] == 'baseline':
        agg[('baseline', None, None, None)].append(r)
    else:
        agg[('block', r['block_size'], r['search_radius'], r['metric'])].append(r)

print(f"{'config':<28}{'mse':>10}{'psnr':>10}{'H(R)':>10}{'time_s':>10}")
for key, items in agg.items():
    method = key[0]
    if method == 'baseline':
        label = 'baseline'
    else:
        _, B, R, m = key
        label = f'block B={B} R={R} {m}'
    mse_v = np.mean([r['mse'] for r in items])
    psnr_v = np.mean([r['psnr'] for r in items])
    ent = np.mean([r['residual_entropy'] for r in items])
    rt = np.mean([r['runtime_s'] for r in items])
    print(f'{label:<28}{mse_v:>10.3f}{psnr_v:>10.2f}{ent:>10.3f}{rt:>10.4f}')

# Save the table as CSV for the report.
import csv
csv_path = RESULT_DIR / 'block_matching_metrics.csv'
with open(csv_path, 'w', newline='') as fh:
    fieldnames = ['clip', 'frame_idx', 'method', 'block_size', 'search_radius',
                  'metric', 'mse', 'psnr', 'residual_entropy', 'runtime_s']
    writer = csv.DictWriter(fh, fieldnames=fieldnames)
    writer.writeheader()
    for r in rows:
        writer.writerow({k: r.get(k) for k in fieldnames})
print(f'wrote {csv_path}')

config                             mse      psnr      H(R)    time_s
baseline                       781.250     19.20     0.158    0.0000
block B=8 R=4 sad                0.000       inf    -0.000    0.0189
block B=8 R=4 mse                0.000       inf    -0.000    0.0193
block B=16 R=4 sad               0.000       inf    -0.000    0.0079
block B=16 R=4 mse               0.000       inf    -0.000    0.0082
block B=16 R=8 sad               0.000       inf    -0.000    0.0279
block B=16 R=8 mse               0.000       inf    -0.000    0.0290
block B=32 R=8 sad             171.875       inf     0.037    0.0166
block B=32 R=8 mse             171.875       inf     0.037    0.0177
wrote result/block_matching_metrics.csv


## 2. Temporal consistency evaluation

Three objective metrics over a clip: per-frame variance-of-Laplacian sharpness, mean L1 first-order temporal difference, and mean L1 second-order temporal acceleration (sensitive to flicker and jitter).

In [6]:
def _laplacian(frame):
    f = frame.astype(np.float32)
    pad = np.pad(f, 1, mode='edge')
    return (pad[:-2, 1:-1] + pad[2:, 1:-1] + pad[1:-1, :-2] + pad[1:-1, 2:] - 4 * f)


def frame_sharpness(clip):
    """Average variance of Laplacian over frames."""
    clip = np.asarray(clip, dtype=np.float32)
    vals = [float(_laplacian(f).var()) for f in clip]
    return float(np.mean(vals))


def temporal_difference(clip):
    """Average L1 distance between consecutive frames, per pixel."""
    clip = np.asarray(clip, dtype=np.float32)
    diffs = np.abs(clip[1:] - clip[:-1])
    return float(diffs.mean())


def temporal_acceleration(clip):
    """Average L1 second-order temporal difference, per pixel."""
    clip = np.asarray(clip, dtype=np.float32)
    if len(clip) < 3:
        return 0.0
    accel = np.abs(clip[2:] - 2 * clip[1:-1] + clip[:-2])
    return float(accel.mean())


def motion_magnitude(clip, block_size=16, search_radius=8):
    """Mean L2 magnitude of block-matching motion vectors averaged over frame pairs."""
    clip = np.asarray(clip, dtype=np.float32)
    mags = []
    for t in range(1, len(clip)):
        mv, _, _ = block_matching(clip[t - 1], clip[t], block_size, search_radius, metric='sad')
        mags.append(float(np.sqrt((mv.astype(np.float32) ** 2).sum(-1)).mean()))
    return float(np.mean(mags)) if mags else 0.0

In [7]:
# Compute objective metrics for every clip and print a table.
eval_rows = []
for name, clip in clips.items():
    eval_rows.append({
        'clip': name,
        'sharpness': frame_sharpness(clip),
        'temporal_difference': temporal_difference(clip),
        'temporal_acceleration': temporal_acceleration(clip),
        'motion_magnitude': motion_magnitude(clip, block_size=16, search_radius=8),
    })

print(f"{'clip':<20}{'sharp':>10}{'tdiff':>10}{'tacc':>10}{'mmag':>10}")
for r in eval_rows:
    print(f"{r['clip']:<20}{r['sharpness']:>10.2f}{r['temporal_difference']:>10.3f}"
          f"{r['temporal_acceleration']:>10.3f}{r['motion_magnitude']:>10.3f}")

import csv
with open(RESULT_DIR / 'consistency_metrics.csv', 'w', newline='') as fh:
    writer = csv.DictWriter(fh, fieldnames=['clip', 'sharpness', 'temporal_difference', 'temporal_acceleration', 'motion_magnitude'])
    writer.writeheader()
    for r in eval_rows:
        writer.writerow(r)

# Subjective ratings (1-5) and one-sentence comment per clip.
human_eval = {
    'clean_translation': dict(visual_quality=5, temporal_consistency=5, motion_naturalness=5,
                              note='clean uniform translation, no visible artifacts.'),
    'flicker':           dict(visual_quality=3, temporal_consistency=2, motion_naturalness=3,
                              note='alternating brightness creates strong flicker between odd/even frames.'),
    'jitter':            dict(visual_quality=3, temporal_consistency=2, motion_naturalness=2,
                              note='non-uniform vertical and horizontal offsets give a visible shaky look.'),
    'blur':              dict(visual_quality=3, temporal_consistency=5, motion_naturalness=4,
                              note='spatial Gaussian blur reduces sharpness but motion remains smooth.'),
}
for k, v in human_eval.items():
    print(k, v)

with open(RESULT_DIR / 'human_eval.csv', 'w', newline='') as fh:
    writer = csv.DictWriter(fh, fieldnames=['clip', 'visual_quality', 'temporal_consistency', 'motion_naturalness', 'note'])
    writer.writeheader()
    for k, v in human_eval.items():
        row = {'clip': k}
        row.update(v)
        writer.writerow(row)

clip                     sharp     tdiff      tacc      mmag
clean_translation       566.41     3.906     7.812     9.883
flicker                 538.79    46.111    91.401     9.429
jitter                  566.41     6.428    12.974    10.051
blur                      5.14     2.561     2.420     9.071
clean_translation {'visual_quality': 5, 'temporal_consistency': 5, 'motion_naturalness': 5, 'note': 'clean uniform translation, no visible artifacts.'}
flicker {'visual_quality': 3, 'temporal_consistency': 2, 'motion_naturalness': 3, 'note': 'alternating brightness creates strong flicker between odd/even frames.'}
jitter {'visual_quality': 3, 'temporal_consistency': 2, 'motion_naturalness': 2, 'note': 'non-uniform vertical and horizontal offsets give a visible shaky look.'}
blur {'visual_quality': 3, 'temporal_consistency': 5, 'motion_naturalness': 4, 'note': 'spatial Gaussian blur reduces sharpness but motion remains smooth.'}


## 3. Bonus: Hierarchical block matching (Option B)

Build a Gaussian pyramid by repeated 2x downsampling, run exhaustive block matching at the coarsest level with a small search radius, then refine each level by upsampling the previous motion field (scaling vectors by 2) and searching only a tiny radius around the prediction. This avoids the quadratic blow-up of the search window.

In [8]:
def build_gaussian_pyramid(frame, levels=3):
    """Coarsest-first list of Gaussian-pyramid frames (levels entries).

    Each level is the previous one blurred with a 5x5 Gaussian and downsampled
    by 2. pyramid[0] is the coarsest, pyramid[-1] is the original.
    """
    kernel = np.array([1, 4, 6, 4, 1], dtype=np.float32) / 16.0
    cur = np.asarray(frame, dtype=np.float32)
    pyr = [cur]
    for _ in range(levels - 1):
        # separable 1D Gaussian blur with reflect padding.
        h = cur
        pad = np.pad(h, ((0, 0), (2, 2)), mode='reflect')
        h = sum(kernel[k] * pad[:, k:k + h.shape[1]] for k in range(5))
        pad = np.pad(h, ((2, 2), (0, 0)), mode='reflect')
        h = sum(kernel[k] * pad[k:k + h.shape[0], :] for k in range(5))
        cur = h[::2, ::2]
        pyr.append(cur)
    return list(reversed(pyr))  # coarsest first


def hierarchical_block_matching(prev_frame, cur_frame, block_size=16, levels=3,
                                coarse_search_radius=4, refine_radius=1, metric='sad'):
    """Coarse-to-fine block matching via Gaussian pyramid.

    The coarsest level uses exhaustive `block_matching` with the requested
    `coarse_search_radius`. Finer levels reuse the previous motion field
    (scaled by 2 in both magnitude and grid) and refine each block within
    `+/- refine_radius` pixels. Block size is held fixed across levels.
    """
    pyr_prev = build_gaussian_pyramid(prev_frame, levels)
    pyr_cur = build_gaussian_pyramid(cur_frame, levels)

    # Coarsest exhaustive search.
    mv, _, _ = block_matching(pyr_prev[0], pyr_cur[0], block_size,
                              search_radius=coarse_search_radius, metric=metric)

    # Refine on each finer level.
    for lvl in range(1, levels):
        prev_l = pyr_prev[lvl]
        cur_l = pyr_cur[lvl]
        H, W = cur_l.shape
        Hb, Wb = H // block_size, W // block_size

        # Upsample motion field: scale vectors by 2, expand grid by 2x in each
        # axis (block size unchanged means grid doubles when image doubles).
        coarse_Hb, coarse_Wb = mv.shape[:2]
        # Map each new (ib, jb) to coarse (ib//2, jb//2).
        new_mv = np.zeros((Hb, Wb, 2), dtype=np.int32)
        for ib in range(Hb):
            for jb in range(Wb):
                cy = min(ib // 2, coarse_Hb - 1)
                cx = min(jb // 2, coarse_Wb - 1)
                new_mv[ib, jb] = mv[cy, cx] * 2

        # Refine each block within +/- refine_radius around the predicted vector.
        refined = np.zeros_like(new_mv)
        block_y = np.arange(Hb) * block_size
        block_x = np.arange(Wb) * block_size
        cur_blocks = cur_l[:Hb * block_size, :Wb * block_size].reshape(
            Hb, block_size, Wb, block_size).transpose(0, 2, 1, 3)

        best_cost = np.full((Hb, Wb), np.inf, dtype=np.float32)
        best_dy = new_mv[..., 0].copy()
        best_dx = new_mv[..., 1].copy()

        for ddy in range(-refine_radius, refine_radius + 1):
            for ddx in range(-refine_radius, refine_radius + 1):
                cand_dy = new_mv[..., 0] + ddy
                cand_dx = new_mv[..., 1] + ddx
                ys = block_y[:, None] + cand_dy
                xs = block_x[None, :] + cand_dx
                valid = (ys >= 0) & (ys + block_size <= H) & (xs >= 0) & (xs + block_size <= W)
                ys_c = np.clip(ys, 0, H - block_size)
                xs_c = np.clip(xs, 0, W - block_size)
                cand = np.empty((Hb, Wb, block_size, block_size), dtype=np.float32)
                for ib in range(Hb):
                    for jb in range(Wb):
                        y0 = ys_c[ib, jb]
                        x0 = xs_c[ib, jb]
                        cand[ib, jb] = prev_l[y0:y0 + block_size, x0:x0 + block_size]
                diff = cur_blocks - cand
                if metric == 'sad':
                    cost = np.sum(np.abs(diff), axis=(2, 3))
                else:
                    cost = np.mean(diff * diff, axis=(2, 3))
                cost = np.where(valid, cost, np.inf)
                improved = cost < best_cost
                best_cost = np.where(improved, cost, best_cost)
                best_dy = np.where(improved, cand_dy, best_dy)
                best_dx = np.where(improved, cand_dx, best_dx)

        refined[..., 0] = best_dy
        refined[..., 1] = best_dx
        mv = refined

    # Build prediction at full resolution.
    H, W = cur_frame.shape
    Hb, Wb = H // block_size, W // block_size
    pred = np.asarray(cur_frame, dtype=np.float32).copy()
    prev_full = np.asarray(prev_frame, dtype=np.float32)
    for ib in range(Hb):
        for jb in range(Wb):
            dy = int(mv[ib, jb, 0])
            dx = int(mv[ib, jb, 1])
            y0 = ib * block_size
            x0 = jb * block_size
            ys = y0 + dy
            xs = x0 + dx
            if 0 <= ys <= H - block_size and 0 <= xs <= W - block_size:
                pred[y0:y0 + block_size, x0:x0 + block_size] = prev_full[ys:ys + block_size, xs:xs + block_size]
            else:
                pred[y0:y0 + block_size, x0:x0 + block_size] = prev_full[y0:y0 + block_size, x0:x0 + block_size]
    residual = np.asarray(cur_frame, dtype=np.float32) - pred
    return mv, pred, residual

In [9]:
# Compare exhaustive vs. hierarchical on the same 5 frame pairs of the clean
# translation clip. We use exhaustive (B=16, R=8) as the high-quality reference
# and hierarchical with 3 pyramid levels, coarse R=4, refine radius=1 (effective
# search radius ~ 4*4 + 1*2 + 1 = 19 px in original-frame coordinates).
bonus_rows = []
for t in range(1, 1 + n_pairs):
    prev_t = clip[t - 1]
    cur_t = clip[t]

    t0 = time.perf_counter()
    _, pred_e, res_e = block_matching(prev_t, cur_t, block_size=16, search_radius=8, metric='sad')
    t_e = time.perf_counter() - t0

    t0 = time.perf_counter()
    mv_h, pred_h, res_h = hierarchical_block_matching(prev_t, cur_t, block_size=16, levels=3,
                                                      coarse_search_radius=4, refine_radius=1, metric='sad')
    t_h = time.perf_counter() - t0

    s_e = summarize_prediction(cur_t, pred_e, res_e)
    s_h = summarize_prediction(cur_t, pred_h, res_h)
    bonus_rows.append({
        'frame_idx': t,
        'exhaustive_psnr': s_e['psnr'], 'exhaustive_mse': s_e['mse'], 'exhaustive_time_s': t_e,
        'hierarchical_psnr': s_h['psnr'], 'hierarchical_mse': s_h['mse'], 'hierarchical_time_s': t_h,
    })

    if t == 1:
        plot_motion_vectors(mv_h, 16, RESULT_DIR / 'hierarchical_mv.png')
        plot_prediction(prev_t, cur_t, pred_h, res_h, RESULT_DIR / 'hierarchical_pred.png')

print(f"{'frame':<8}{'exh PSNR':>12}{'exh time':>12}{'hier PSNR':>12}{'hier time':>12}")
for r in bonus_rows:
    print(f"{r['frame_idx']:<8}{r['exhaustive_psnr']:>12.2f}{r['exhaustive_time_s']:>12.4f}"
          f"{r['hierarchical_psnr']:>12.2f}{r['hierarchical_time_s']:>12.4f}")

exh_p = np.mean([r['exhaustive_psnr'] for r in bonus_rows])
hier_p = np.mean([r['hierarchical_psnr'] for r in bonus_rows])
exh_t = np.mean([r['exhaustive_time_s'] for r in bonus_rows])
hier_t = np.mean([r['hierarchical_time_s'] for r in bonus_rows])
print(f'mean: exhaustive PSNR={exh_p:.2f} time={exh_t:.4f}s  |  hierarchical PSNR={hier_p:.2f} time={hier_t:.4f}s')

with open(RESULT_DIR / 'hierarchical_vs_exhaustive.csv', 'w', newline='') as fh:
    writer = csv.DictWriter(fh, fieldnames=list(bonus_rows[0].keys()))
    writer.writeheader()
    for r in bonus_rows:
        writer.writerow(r)

frame       exh PSNR    exh time   hier PSNR   hier time
1                inf      0.0286       32.55      0.0048
2                inf      0.0287       32.51      0.0049
3                inf      0.0287       32.81      0.0048
4                inf      0.0285       33.88      0.0048
5                inf      0.0299       43.89      0.0048
mean: exhaustive PSNR=inf time=0.0289s  |  hierarchical PSNR=35.13 time=0.0048s
